# Many-Body Fractional Charge Pump
## One-dimensional flux-cylinder diagnostic for interacting topological phases

This notebook provides a self-contained pedagogical derivation and hands-on guide to the **fractional charge pump** implemented in `observables/charge_pump.jl`.  The calculation threads a flux $\theta$ through one periodic direction of a finite torus and tracks the winding of the many-body polarization in the transverse direction — a finite-size analogue of the Laughlin–Thouless charge pump.

**It is deliberately *not* a two-dimensional many-body Chern-number calculation** (which would require a 2D flux-torus grid).  The 1D pump is faster, more intuitive, and captures the same universal fractional Hall response.

The key design innovation is the **flux-aware symmetry group**: rather than diagonalizing the full Hilbert space at each flux value, we keep the calculation in symmetry-resolved momentum sectors whose labels remain invariant under flux insertion.  This gives orders-of-magnitude speedup for moderate system sizes.

## Physical Setup: Flux Cylinder and Laughlin Pump

Consider a two-dimensional periodic finite system (a torus) with $L_x \times L_y$ unit cells.  We thread an adiabatic flux $\theta(t)$ along the $x$-direction.  In the non-interacting integer quantum Hall regime, Laughlin's gauge argument shows that each occupied band with Chern number $C$ pumps $C$ charges across any $y$-section per flux quantum.  For strongly-correlated fractional phases, the same flux-threading pumps a *fractional* charge — the many-body Hall conductance in units of $e^2/h$.

Our computational setup:

- **Flux insertion**: $\theta_x$ multiplexes the hopping terms crossing the $x$-periodic boundary by a Peierls phase $e^{i 2\pi \theta_x}$.
- **Polarization measurement**: In the transverse ($y$) direction we define the periodic many-body position following Resta:

\begin{equation}
\boxed{\hat U_y = \exp\!\left(\frac{2\pi i}{L_y}\sum_{j=1}^{N} x_{j,y}\,\hat n_j\right)},
\end{equation}

where $x_{j,y}$ is the crystal-coordinate position of site $j$ along the $y$-direction, measured in unit-cell units, including the sublattice offset $\delta_{y}(\text{sub}_j)$ by default.  $L_y$ is the number of unit cells in that direction.

The operator $\hat U_y$ is unitary and periodic: $\hat U_y^{L_y} = \mathbb{1}$.  Its eigenvalues are phases $e^{i 2\pi P}$ where $P \in [0,1)$ is the **many-body polarization**.  The charge pumped across a $y$-cut when $\theta_x$ advances by one flux quantum is:

\begin{equation}
\boxed{\Delta Q = P(\theta_x = 1) - P(\theta_x = 0)} \qquad (\text{in units of } e).
\end{equation}

For a fractional Chern insulator at filling $\nu = p/q$, one expects $q$ nearly-degenerate ground states on the torus (the topological ground-state multiplet).  Threading one flux quantum cyclically permutes these states, and each carries a polarization winding of $\Delta Q = 1/q$.  The total pumped charge $\sum_i \Delta Q_i = 1$ is integer — the many-body Chern number.

## Flux-Aware Symmetry: Why Momentum Labels Stay Fixed

A crucial subtlety: under flux insertion the Hamiltonian $H(\theta)$ does **not** commute with the ordinary translation operator $T^0$.  The boundary-gauge Peierls phase breaks naive translation invariance.  The correct translation that *does* commute with $H(\theta)$ is the **gauge-covariant translation**:

\begin{equation}
T^{\theta}_{(d_x,d_y)} = G_\theta\, T^0_{(d_x,d_y)}\, G_\theta^{-1}, \qquad
G_\theta = \exp\!\left(i 2\pi\sum_{j} \frac{\theta\cdot x_j}{L} \hat n_j\right).
\end{equation}

$T^\theta$ has the same eigenvalue spectrum as $T^0$ — the standard crystal momenta $[k_1,k_2]$ — because it is unitarily equivalent.  In our implementation, the `perm_phases` field of each `Symmetry_Operation` absorbs the gauge transformation, keeping the **irrep labels unchanged** under flux.  The orbit-stabilizer catalog is built once at $\theta = 0$; at each subsequent $\theta$ only the stabilizer phases are updated via `update_orbit_stabilizer_phases!`, avoiding the full $O(\binom{N}{N_e})$ Gosper enumeration.

## Symmetry-Resolved Projected Position Operator

The polarization phases come from diagonalizing $\hat U_y$ in the low-energy manifold.  Because $\hat U_y$ is **diagonal** in the Fock basis, $\hat U_y|m\rangle = u_y(m)|m\rangle$ with

\begin{equation}
u_y(m) = \prod_{j \in \text{occ}(m)} e^{2\pi i x_{j,y}/L_y},
\end{equation}

its matrix elements between symmetry-projected basis states factorize cleanly.

Recall from `design.ipynb` that the normalized symmetry-projected state for orbit representative $[s]$ and irrep $\chi$ is:

\begin{equation}
|\widetilde{[s];\chi}\rangle = \sqrt{\frac{|\text{Stab}(s)|}{|G|}} \sum_{g \in G/\text{Stab}(s)} \chi(g)^*\, U_g |[s]\rangle.
\end{equation}

For a diagonal operator $D|m\rangle = d(m)|m\rangle$, the matrix element between irreps $\chi_{\text{to}}$ and $\chi_{\text{from}}$ on the **same** orbit representative is:

\begin{equation}
\boxed{\langle \widetilde{[s];\chi_{\text{to}}}| D |\widetilde{[s];\chi_{\text{from}}}\rangle = \frac{1}{|G|} \sum_{g \in G} \chi_{\text{to}}(g)\, \chi_{\text{from}}(g)^*\, d(g\cdot s)}.
\end{equation}

Representatives from different orbits give exactly zero because $D$ cannot connect different orbits (it is diagonal and each orbit is a disjoint set of Fock states).  The prefactor $1/|G|$ (rather than $\sqrt{\text{Stab}}$ factors) comes from the normalization-convention cancellation detailed in `design.ipynb`.

Our implementation in `_position_operator_matrix` computes the sparse CSC matrix of $\hat U_y$ in this inter-sector representation, then projects it into the low-energy eigenstate basis of $H(\theta)$.

## Algorithmic Walkthrough

At each flux value $\theta$ in the scan list, `flux_charge_pump` performs the following steps:

### Step 1: Update the Model In-Place

```julia
update_second_quantized_model_with_twisted_phases!(
    model; twisted_phases_over_2π = [θ, 0.0])
```

This calls `TightBinding.generate_bilinear_terms` with the current flux, replacing the model's `bilinear_terms` vector.  No new model object is allocated — the same `model` is mutated throughout the scan.

### Step 2: Update the Flux-Aware Symmetry Group

```julia
active_group = build_translation_group(model.lattice, [θ, 0.0])
ed_data.symmetry_group = active_group
update_orbit_stabilizer_phases!(ed_data.orbit_catalog, active_group, model.statistics)
```

The orbit *partition* (which Fock states belong to which orbit) is invariant — only the stabilizer phases need recomputation.  This costs $O(N_{\text{orbits}} \cdot |G|)$, which is negligible compared to the initial $O(\binom{N}{N_e})$ enumeration.

### Step 3: Diagonalize & Build Sector Bases

For each requested momentum sector, diagonalize the interacting Hamiltonian (via Arpack on the sparse CSC block) and build the symmetry-sector basis:

```julia
vals, vecs = ed_scan_at_irrep_matrix!(label, ed_data; nev=nev_per_sector)
irrep, _ = _irrep_for_label(ed_data, label)
bases[label] = build_symmetry_sector_basis(ed_data.orbit_catalog, irrep)
```

### Step 4: Project $\hat U_y$ into the Low-Energy Manifold

The periodic position operator matrix $U_{y}^{\alpha\beta} = \langle \widetilde{[s];\chi_\alpha} | \hat U_y | \widetilde{[s];\chi_\beta} \rangle$ is built between sector bases via `_position_operator_matrix`, then projected into the low-energy eigenstates:

\begin{equation}
P_{\alpha\beta}(\theta) = \sum_{a,b} \big(\psi^{(\alpha)}_a\big)^\dagger \; U_y^{\alpha\beta} \; \psi^{(\beta)}_b,
\end{equation}

where $\psi^{(\alpha)}_a$ is the $a$-th low-energy eigenvector in sector $\chi_\alpha$.  $P(\theta)$ is an $n_{\text{states}} \times n_{\text{states}}$ matrix representing $\hat U_y$ restricted to the low-energy manifold.

### Step 5: Unwrap Phases & Extract Pumped Charge

The eigenvalues $\lambda_i(\theta)$ of $P(\theta)$ are complex phases.  We sort them by argument at each $\theta$ and unwrap the branches using optimal permutation matching to minimize discontinuities:

\begin{equation}
P_i(\theta) = \frac{1}{2\pi} \arg \lambda_i(\theta) \quad \text{(unwrapped)},
\qquad
\Delta Q_i = P_i(\theta_{\max}) - P_i(0).
\end{equation}

The result stores both `polarizations` (raw unwrapped phases) and `pumped_charge_trajectories` (shifted to start from zero for plotting).

### Code Skeleton

```julia
# Core call — all steps above are internal
pump = flux_charge_pump(
    model,
    [(0, 0), (1, 0)];                  # the topological multiplet
    filling_fraction      = 1 // 4,     # 3 bosons / 12 vertices
    flux_direction        = 1,          # θ_x flux
    polarization_direction = 2,         # measure U_y
    twisted_phases_list   = collect(range(0.0, 1.0; length=9)),
    nev_per_sector        = 1,          # one state per sector
    fig_path              = "figures/charge_pump.svg",
    checkpoint_path       = "checkpoints/charge_pump.jld2",
)

pump.pumped_charges  # e.g. [0.5, 0.5] for the ν=1/2 FCI
```


## Hands-On: Bosonic Haldane FCI on $[2,3]$ Honeycomb

The Haldane honeycomb model at half filling of the lower Chern band hosts a bosonic fractional Chern insulator at $\nu = 1/2$ (D.N. Sheng *et al.*, PRL **107**, 146803, 2011).  On a $2 \times 3$ torus (12 sites, 3 hard-core bosons), the interacting ground state is the finite-size analogue of the bosonic Laughlin state, with a topological ground-state degeneracy of $2$ on the torus.

### Model Construction

```julia
using RealSpace_ExactDiagonalization

# Parameters from Sheng et al.
model = build_zero_flux_bosonic_fci_second_quantized_model(;
    sample_size = [2, 3],
    params = Dict(
        "t"  => 1.0,      # NN hopping
        "t′" => 0.60,     # NNN hopping amplitude
        "t′′"=> -0.58,    # 3rd-NN hopping
        "ϕ_over_2π" => 0.2,  # Haldane flux 0.4π
        "V1" => 0.0,      # NN interaction (set to 0 for FCI)
        "V2" => 0.0,      # NNN interaction
    ),
)
```

The two FCI ground states at $\theta = 0$ reside in momentum sectors $[0,0]$ and $[1,0]$:

```julia
labels = default_fci_sectors([2, 3])  # returns [(0,0), (1,0)]
```

### Running the Charge Pump

```julia
pump = flux_charge_pump(
    model,
    labels;
    filling_fraction      = 1 // 4,    # n_filled / n_vertices
    flux_direction        = 1,          # thread θ_x
    polarization_direction = 2,         # measure U_y polarization
    twisted_phases_list   = collect(range(0.0, 1.0; length = 9)),
    nev_per_sector        = 1,
    fig_path              = "figures/bosonic_FCI_charge_pump.svg",
    checkpoint_path       = "checkpoints/bosonic_FCI_charge_pump.jld2",
)
```

### Self-Check (with `Test`)

```julia
using Test
test_bosonic_fci_charge_pump(;
    sample_size          = [2, 3],
    twisted_phases_list  = collect(range(0.0, 1.0; length = 9)),
    mode                 = :sectors,
)
```

### Expected Results

The two polarization branches each wind by **one half charge** over a single flux quantum:

| Polarization branch | $\Delta Q$ (pumped charge) |
|---------------------|---------------------------|
| Branch 1 (from sector $[0,0]$) | $\approx 0.5$ |
| Branch 2 (from sector $[1,0]$) | $\approx 0.5$ |
| **Total** | $\mathbf{1.0}$ (integer) |

This is the **semion topological order** — the finite-size fingerprint of the $\nu = 1/2$ bosonic Laughlin state.  The spectrum-flow observable (in `spectrum_flow.jl`) sees the same physics as a cyclic exchange of the two ground states: the $[0,0]$ GS adiabatically evolves into the $[1,0]$ GS after one flux quantum and returns after two.

### Interpretation

The two branches correspond to the two topological ground states on the torus.  Under $2\pi$ flux insertion (the $T$-transformation of the modular group), the anyon excitations transform projectively — each charge $e/2$ semion contributes a polarization winding of $1/2$.  Different sample sizes give different GSD patterns according to $n_{\text{filled}} \bmod L_x$ (the commensurability condition).

## References

1. **Resta's polarization**: R. Resta, *Quantum-mechanical position operator in extended systems*, Phys. Rev. Lett. **80**, 1800 (1998).
2. **Laughlin charge pump**: R. B. Laughlin, *Quantized Hall conductivity in two dimensions*, Phys. Rev. B **23**, 5632 (1981).
3. **Bosonic FCI model**: D. N. Sheng, Z.-C. Gu, K. Sun, & L. Sheng, *Fractional Chern insulator on the honeycomb lattice with bosons*, Phys. Rev. Lett. **107**, 146803 (2011).
4. **Many-body polarization for fractional Chern insulators**: T. Neupert, L. Santos, C. Chamon, & C. Mudry, *Fractional Chern insulators*, Phys. Rev. Lett. **106**, 236804 (2011).
5. **ExactDiagonalization.jl showcase**: [Quantum-Many-Body/ExactDiagonalization.jl](https://github.com/Quantum-Many-Body/ExactDiagonalization.jl) — the reference implementation that inspired our spectral-flow and charge-pump diagnostics.
6. **XDiag**: [awietek/xdiag](https://github.com/awietek/xdiag) — the symmetry-resolved ED framework whose orbit-stabilizer design philosophy we follow.
7. **Symmetry-resolved ED design**: `doc/design.ipynb` in this repository — the companion notebook deriving the bitmask encoding, orbit-stabilizer decomposition, and irrep projection.

---

*This notebook is part of the `RealSpace_ExactDiagonalization.jl` package.  The implementation lives in `observables/charge_pump.jl` (core method) and `test/bosonic_fci.jl` (self-contained test).*